In [228]:
import os
import json
import pandas as pd
import traceback

In [229]:
from langchain.chat_models import ChatOpenAI

In [230]:
# from dotenv import load_dotenv
# load_dotenv()  # take environment variables from .env.

In [231]:
KEY="sk-proj-xOp2svyhOMJQH9k1RzjeT3BlbkFJpjgY2YmiumpX8jRKJ3bU"
KEY

'sk-proj-xOp2svyhOMJQH9k1RzjeT3BlbkFJpjgY2YmiumpX8jRKJ3bU'

In [232]:
llm=ChatOpenAI(openai_api_key=KEY, model_name="gpt-3.5-turbo", temperature=0.5)

In [233]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import PyPDF2

In [234]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [235]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz  of {number} multiple choice questions for {subject} students in {tone} tone. 
Make sure the questions are not repeated and check all the questions to be conforming the text as well.
Make sure to format your response like  RESPONSE_JSON below  and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [236]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
    )

In [237]:
quiz_chain=LLMChain(llm=llm,prompt=quiz_generation_prompt,output_key="quiz",verbose=True)

In [238]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [239]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject","quiz"], template=TEMPLATE)

In [240]:
review_chain=LLMChain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [241]:
generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review_chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [242]:
file_path=r"/workspaces/mcqgen/data.txt"
file_path

'/workspaces/mcqgen/data.txt'

In [243]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [244]:
# Serialize the Python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [245]:
NUMBER=5 
SUBJECT="GalaxyZFlips"
TONE="medium difficulty"

In [246]:
#https://python.langchain.com/docs/modules/model_io/llms/token_usage_tracking

#How to setup Token Usage Tracking in LangChain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject":SUBJECT,
            "tone": TONE,
            "response_json": json.dumps(RESPONSE_JSON)
        }
        )



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

Text:Galaxy Z Flip6


Galaxy Flip6 Key Features: Camera

BEST FLIP CAMERA
•	The brightest, clearest & high-resolution Selfies, Photos & Videos - 50MP Wide Lens, Bigger 2.0µm Pixel
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

FLEXCAM PHOTOGRAPHY
•	Take hands-free Photos, videos & Portraits without touching the Phone using Cover screen - FlexCam – Hands-free
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

AI AUTO ZOOM
•	Camera Auto zooms in & out to get the best angle for your Awesome Photoshoot.
•	AI detects subject & background
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

AI GENERATIVE PORTRAITS
•	Best portraits Powered by ProVisual Engine
•	Clear outlines & authentic facial expressions - Object Aware Engine
•	DSLR-like natural bokeh effect for your Portraits - AI Stereo Depth

BEST NIGHTOGRAPHY
•	Enhanced noise r


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:

Text:Galaxy Z Flip6


Galaxy Flip6 Key Features: Camera

BEST FLIP CAMERA
•	The brightest, clearest & high-resolution Selfies, Photos & Videos - 50MP Wide Lens, Bigger 2.0µm Pixel
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

FLEXCAM PHOTOGRAPHY
•	Take hands-free Photos, videos & Portraits without touching the Phone using Cover screen - FlexCam – Hands-free
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

AI AUTO ZOOM
•	Camera Auto zooms in & out to get the best angle for your Awesome Photoshoot.
•	AI detects subject & background
•	Use Watch7 Camera Controller for TRUE hands-free Camera experience

AI GENERATIVE PORTRAITS
•	Best portraits Powered by ProVisual Engine
•	Clear outlines & authentic facial expressions - Object Aware Engine
•	DSLR-like natural bokeh effect for your Portraits - AI Stereo Depth

BEST NIGHTOGRAPHY
•	Enhanced noise reduction to capture the

In [247]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

Total Tokens:13695
Prompt Tokens:12872
Completion Tokens:823
Total Cost:0.020954


In [248]:
response

{'text': 'Galaxy Z Flip6\n\n\nGalaxy Flip6 Key Features: Camera\n\nBEST FLIP CAMERA\n•\tThe brightest, clearest & high-resolution Selfies, Photos & Videos - 50MP Wide Lens, Bigger 2.0µm Pixel\n•\tUse Watch7 Camera Controller for TRUE hands-free Camera experience\n\nFLEXCAM PHOTOGRAPHY\n•\tTake hands-free Photos, videos & Portraits without touching the Phone using Cover screen - FlexCam – Hands-free\n•\tUse Watch7 Camera Controller for TRUE hands-free Camera experience\n\nAI AUTO ZOOM\n•\tCamera Auto zooms in & out to get the best angle for your Awesome Photoshoot.\n•\tAI detects subject & background\n•\tUse Watch7 Camera Controller for TRUE hands-free Camera experience\n\nAI GENERATIVE PORTRAITS\n•\tBest portraits Powered by ProVisual Engine\n•\tClear outlines & authentic facial expressions - Object Aware Engine\n•\tDSLR-like natural bokeh effect for your Portraits - AI Stereo Depth\n\nBEST NIGHTOGRAPHY\n•\tEnhanced noise reduction to capture the detail with the dedicated ISP (Image si

In [249]:
quiz = response.get('quiz')
quiz = json.loads(quiz)

In [250]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [251]:
quiz_table_data

[{'MCQ': 'What is the key feature of the Galaxy Z Flip6 camera that allows you to take hands-free photos, videos, and portraits without touching the phone?',
  'Choices': 'a: FlexCam Photography | b: AI Generative Portraits | c: AI Auto Zoom | d: Best Nightography',
  'Correct': 'a'},
 {'MCQ': 'Which Galaxy Z Flip6 feature allows you to select any content on the screen to view creative AI actions using Smart Select from the edge panel?',
  'Choices': 'a: Sketch to Image | b: Photo Assist Editing | c: Portrait Studio | d: Smart Select',
  'Correct': 'd'},
 {'MCQ': 'What Galaxy Z Flip6 feature provides an AI-powered editing tool to create portraits in various styles such as Comic, Watercolor, 3D Cartoon, and Sketch?',
  'Choices': 'a: Live Effect | b: Portrait Studio | c: Interactive 3D Emoji Wallpaper | d: Wallpaper Suggestion',
  'Correct': 'b'},
 {'MCQ': 'Which Galaxy Z Flip6 feature allows you to translate conversations in real-time and view them as text, supporting offline translati

In [252]:
quiz = pd.DataFrame(quiz_table_data)

In [253]:
quiz.to_csv("GalaxyZFlip6.csv",index=False)